In [2]:
import pandas as pd
import altair as alt
import json
import vl_convert as vlc
import time
from PIL import Image

In [3]:
df=pd.DataFrame({'Org': ['Social 1&2','Social 1&2','Social 1&2','Social 1&2', 'Social 1&2','Recall 1&2', 'Recall 1&2', 'Recall 1&2', 'Recall 1&2', 'Recall 1&2','Recall 3&4', 'Recall 3&4', 'Recall 3&4', 'Recall 3&4','Recall 3&4'],
                 'Response profile': ['CS-', 'CS+', 'Mixed', 'Non-soc', 'Non','CS-', 'CS+', 'Mixed', 'Non-soc','Non', 'CS-', 'CS+', 'Mixed', 'Non-soc','Non'],
                'numbers': [7,9,10,22,52,14,13,8,24,41,12,15,9,21,43]})

#---1. increase distance between bars -- now they are overlapping. Done
# Change figure size OR/AND
# Change bar size
bar_size = 50

#2, scale bar below bars from 0 to 100.
# Adjust numbers

#3. names of legend (rose etc. should be changed to other names) changeable
# Since Altair legends reflect the content of the data. The cleanest way to change the values they show is to change the values within the dataset
# Also set color for values here

## colors
# purple #6e05ab
# light blue #2462bf
# lightgrey
# blue-purple: #3f3999
value_map = {
    'CS+' : ['CS+ selective','#6e05ab'],
    'Non' : ['Other','lightgrey'],
    'Non-soc' : ['Spatially selective','white'],
    'Mixed' : ['Mixed selectivity','#3f3999'],
    'CS-' : ['CS- selective','#2462bf'],
}

df['Response profile'] = df['Response profile'].apply(lambda x: value_map[x][0])
domain = [i[0] for i in value_map.values()]
range = [i[1] for i in value_map.values()]

#4. larger font size numbers on scale below the bars
#5. larger fontsize for legend text and larger legend in general

# Set font sizes
axis_labelFontSize = 22
title_FontSize = 22
text_FontSize = 22
legend_labelFontSize = 22
legend_titleFontSize = 22

# # Need to specify the order of bars, follow the name order of Org column
# sort_order = df['Org'].to_list() 

# Or you can specify the order manually:
sort_order = ['Social 1&2','Recall 1&2','Recall 3&4']



### Direction of bars
direction = 'horizontal'  # or 'vertical'
#direction = 'vertical'  # or 'vertical'

if direction == 'horizontal':
    ascending=[True, True]

    x_bar_str = 'sum(numbers)'
    x_text_str = 'position'
    y_bar_str = 'Org'
    y_text_str = 'Org'
    width=400
    height=240
    x_title = 'Fraction of cells (%)'
    y_title = None
else:
    ascending=[True, False]
    x_bar_str = 'Org'
    y_bar_str = 'sum(numbers)'
    x_text_str = 'Org'
    y_text_str = 'position'
    width=240
    height=360
    x_title = None
    y_title = 'Fraction of cells (%)'



# Need to calculate correct postion to show text (middle of bars)
df.sort_values(by=['Org','Response profile'],ascending=ascending,inplace=True)
df['cul'] = df.groupby(['Org'])['numbers'].cumsum()
df['position'] = df.apply(lambda row: row['cul'] - row['numbers'] // 2,axis=1)

### Plot
bars=alt.Chart().mark_bar(stroke='black', strokeWidth=4, opacity=1, fillOpacity = 0.3, size=bar_size).encode(
    x=alt.X(x_bar_str, title='Fraction of cells (%)'),
    y=alt.Y(y_bar_str, title=None, sort=sort_order),
    color=alt.Color('Response profile', title=None, 
                    scale=alt.Scale(domain=domain,
                                    range=range),
                    legend=alt.Legend(title='', orient = 'right',labelFontSize=legend_labelFontSize,titleFontSize=legend_titleFontSize))
)

text=alt.Chart().mark_text(align='center', baseline='middle', size=text_FontSize).encode(
    y=alt.Y(y_text_str, title=None, sort=sort_order),
    x=x_text_str,
    text='numbers')

chart_out = alt.layer(bars, text, data=df).properties(
    width=360,
    height=240
).configure_axis(
    labelFontSize = axis_labelFontSize,
    titleFontSize = title_FontSize,
).configure_legend(labelLimit= 0, titleLimit=0)

chart_out

alt.LayerChart(...)

In [4]:
# Save as jpg and eps
# No direct way to save an altair chart to desired format.
# json -> png -> jpg & eps
chart_out.save('temp_json.json')
with open('temp_json.json') as f:
    temp_json = json.load(f)

time.sleep(5)
png_data = vlc.vegalite_to_png(vl_spec=temp_json, scale=2)
with open('chart.png', "wb") as f:
    f.write(png_data)

im = Image.open(r'chart.png')
rgb_im = im.convert('RGB')
rgb_im.save(r'BarplotSFC.jpg')
rgb_im.save('BarplotSFC.eps')